In [1]:
import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine

#### Conexion con las bases de datos

In [2]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

config_source = config["mensajeria_bd"]
config_dw = config["etl_mensajeria"]

In [3]:
url_source = (
    f"{config_source['drivername']}://"
    f"{config_source['user']}:{config_source['password']}@"
    f"{config_source['host']}:{config_source['port']}/"
    f"{config_source['dbname']}"
)

url_dw = (
    f"{config_dw['drivername']}://"
    f"{config_dw['user']}:{config_dw['password']}@"
    f"{config_dw['host']}:{config_dw['port']}/"
    f"{config_dw['dbname']}"
)

engine_source = create_engine(url_source)
engine_dw = create_engine(url_dw)

#### Extraer datos

In [4]:
# Extraer estados del servicio
df_estados = pd.read_sql_table(
    "mensajeria_estadosservicio",
    engine_source
)

df_estados.head()

,id,fecha,hora,foto,observaciones,estado_id,servicio_id,es_prueba,foto_binary
0,1014,2024-01-29,01:13:32,foto,4 tubos,4,226,False,None
1,1484,2024-01-30,18:45:12,foto,Demora,3,79,True,None
2,2829,2024-02-06,11:34:04,foto,Compra exitosa,5,613,False,None
3,1888,2024-02-01,14:50:39,foto,Zzxzz,4,376,False,None
4,32312,2024-04-06,16:11:21,foto,No,3,7164,True,None


In [5]:
# Extraer servicios para obtener mensajero_id
df_servicios = pd.read_sql_table(
    "mensajeria_servicio",
    engine_source
)

df_servicios[["id", "mensajero_id"]].head()

,id,mensajero_id
0,34,NaN
1,35,7.0
2,36,NaN
3,41,NaN
4,42,NaN


In [6]:
# Ver columnas
df_estados.columns

Index(['id', 'fecha', 'hora', 'foto', 'observaciones', 'estado_id',
       'servicio_id', 'es_prueba', 'foto_binary'],
      dtype='str')

#### Transformar

In [7]:
df = df_estados.copy()

# Traer mensajero_id desde mensajeria_servicio
df = df.merge(df_servicios[["id", "mensajero_id"]], left_on="servicio_id", right_on="id", how="left")
df["mensajero_id"] = df["mensajero_id"].fillna(0).astype(int)

# Combinar fecha y hora en un datetime
df["datetime"] = pd.to_datetime(df["fecha"].astype(str) + " " + df["hora"].astype(str), format="mixed")

# Ordenar por servicio y datetime
df = df.sort_values(["servicio_id", "datetime"]).reset_index(drop=True)

# Calcular datetime_fin: inicio del siguiente estado del mismo servicio
df["datetime_fin"] = df.groupby("servicio_id")["datetime"].shift(-1)

# Duracion en minutos; ultimo estado (sin fin) -> 0 para no afectar promedios
df["duracion_tiempo_estado"] = (
    (df["datetime_fin"] - df["datetime"])
    .dt.total_seconds()
    .div(60)
    .fillna(0)
    .astype(int)
)

df[["id_x", "estado_id", "mensajero_id", "datetime", "datetime_fin", "duracion_tiempo_estado", "servicio_id"]].head(10)

,id_x,estado_id,mensajero_id,datetime,datetime_fin,duracion_tiempo_estado,servicio_id
0,6,1,1,2023-09-19 16:22:18.000,2023-10-13 17:51:20.000,34649,7
1,117,2,1,2023-10-13 17:51:20.000,2023-10-31 12:02:48.844,25571,7
2,134,4,1,2023-10-31 12:02:48.844,2023-10-31 12:16:00.000,13,7
3,136,6,1,2023-10-31 12:16:00.000,2023-10-31 17:07:55.000,291,7
4,135,5,1,2023-10-31 17:07:55.000,NaT,0,7
5,7,1,7,2023-09-19 16:30:05.000,2023-12-20 20:14:43.000,132704,8
6,241,2,7,2023-12-20 20:14:43.000,2024-02-14 15:34:18.000,80359,8
7,5355,4,7,2024-02-14 15:34:18.000,2024-04-09 16:08:35.000,79234,8
8,34061,5,7,2024-04-09 16:08:35.000,NaT,0,8
9,8,1,7,2023-09-19 16:30:05.000,2023-12-28 19:33:01.000,144182,9


In [8]:
# Construir dim_tiempo para el join
dim_tiempo = pd.DataFrame({
    "fecha": pd.date_range(start="09/19/2023", end="31/08/2024", freq="D")
})
dim_tiempo["id_tiempo"] = range(1, len(dim_tiempo) + 1)
dim_tiempo["fecha"] = pd.to_datetime(dim_tiempo["fecha"])

# Join para id_tiempo_estado_inicio
df["fecha_inicio"] = df["datetime"].dt.normalize()
df = df.merge(dim_tiempo[["id_tiempo", "fecha"]], left_on="fecha_inicio", right_on="fecha", how="left")
df = df.rename(columns={"id_tiempo": "id_tiempo_estado_inicio"})

# Join para id_tiempo_estado_fin
df["fecha_fin"] = df["datetime_fin"].dt.normalize()
df = df.merge(dim_tiempo[["id_tiempo", "fecha"]], left_on="fecha_fin", right_on="fecha", how="left")
df = df.rename(columns={"id_tiempo": "id_tiempo_estado_fin"})

# Ultimo estado sin fin -> id_tiempo = 1 (valor minimo, no afecta promedios)
df["id_tiempo_estado_fin"] = df["id_tiempo_estado_fin"].fillna(1).astype(int)

# Construir hecho final
hecho = pd.DataFrame()
hecho["id_estado_servicio"]      = df["id_x"]
hecho["id_estado"]               = df["estado_id"]
hecho["id_mensajero"]            = df["mensajero_id"]
hecho["id_tiempo_estado_inicio"] = df["id_tiempo_estado_inicio"].astype(int)
hecho["id_tiempo_estado_fin"]    = df["id_tiempo_estado_fin"]
hecho["duracion_tiempo_estado"]  = df["duracion_tiempo_estado"]
hecho["cod_servicio"]            = df["servicio_id"]

hecho.head()

,id_estado_servicio,id_estado,id_mensajero,id_tiempo_estado_inicio,id_tiempo_estado_fin,duracion_tiempo_estado,cod_servicio
0,6,1,1,1,25,34649,7
1,117,2,1,25,43,25571,7
2,134,4,1,43,43,13,7
3,136,6,1,43,43,291,7
4,135,5,1,43,1,0,7


#### Validar

In [9]:
hecho.info()

<class 'pandas.DataFrame'>
RangeIndex: 128402 entries, 0 to 128401
Data columns (total 7 columns):
 #   Column                   Non-Null Count   Dtype
---  ------                   --------------   -----
 0   id_estado_servicio       128402 non-null  int64
 1   id_estado                128402 non-null  int64
 2   id_mensajero             128402 non-null  int64
 3   id_tiempo_estado_inicio  128402 non-null  int64
 4   id_tiempo_estado_fin     128402 non-null  int64
 5   duracion_tiempo_estado   128402 non-null  int64
 6   cod_servicio             128402 non-null  int64
dtypes: int64(7)
memory usage: 6.9 MB


In [10]:
hecho.describe()

,id_estado_servicio,id_estado,id_mensajero,id_tiempo_estado_inicio,id_tiempo_estado_fin,duracion_tiempo_estado,cod_servicio
count,128402.000000,128402.000000,128402.000000,128402.000000,128402.000000,128402.000000,128402.000000
mean,64391.032764,3.140574,26.563177,248.600762,193.907914,97.240012,14290.879815
std,37184.427582,1.672393,12.490133,59.872343,115.648252,1847.885167,8239.462349
min,6.000000,1.000000,0.000000,1.000000,1.000000,0.000000,7.000000
25%,32180.250000,2.000000,16.000000,201.000000,148.000000,0.000000,7118.250000
50%,64395.500000,3.000000,28.000000,250.000000,224.000000,11.000000,14314.000000
75%,96573.750000,5.000000,34.000000,298.000000,284.000000,37.000000,21426.000000
max,128809.000000,6.000000,84.000000,348.000000,348.000000,198893.000000,28468.000000


In [11]:
hecho.isnull().sum()

id_estado_servicio         0
id_estado                  0
id_mensajero               0
id_tiempo_estado_inicio    0
id_tiempo_estado_fin       0
duracion_tiempo_estado     0
cod_servicio               0
dtype: int64

#### Verificar tiempos

In [12]:
hecho["id_tiempo_estado_inicio"].min()

np.int64(1)

In [13]:
hecho["id_tiempo_estado_inicio"].max()

np.int64(348)

In [14]:
hecho["duracion_tiempo_estado"].describe()

count    128402.000000
mean         97.240012
std        1847.885167
min           0.000000
25%           0.000000
50%          11.000000
75%          37.000000
max      198893.000000
Name: duracion_tiempo_estado, dtype: float64

#### Cargar al Data Warehouse

In [15]:
hecho.to_sql(
    name="hecho_seguimiento_estado",
    con=engine_dw,
    schema="data_mart_entregas",
    if_exists="replace",
    index=False
)

print(f"Cargados {len(hecho)} registros en data_mart_entregas.hecho_seguimiento_estado")

Cargados 128402 registros en data_mart_entregas.hecho_seguimiento_estado
